# PR Review Agent Training on Colab

This notebook runs the PR Review RL training pipeline on Google Colab.

It is designed for a conservative first pass:

- install the repo and training dependencies
- run deterministic sanity checks with `PR_REVIEW_TOOL_BACKEND=heuristic`
- build replayable training states
- run a small GRPO training job
- evaluate the resulting checkpoint
- package artifacts for download

Recommended runtime:

- **T4 GPU** for smoke training
- **A100 GPU** for a fuller run

Before starting a long run, use **Runtime > Change runtime type > GPU**.


## 1. Runtime Check

Confirm that Colab sees a GPU. If this prints `False`, switch to a GPU runtime before training.


In [ ]:
import os, sys, subprocess, textwrap, json, pathlib

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch not installed yet:", exc)

print("Python:", sys.version)


## 2. Get the Repository

Set `REPO_URL` to your GitHub repository URL, then run the cell. If the repo is private, authenticate first or upload a zip manually and adjust `PROJECT_DIR`.


In [ ]:
REPO_URL = ""  # Example: "https://github.com/<user>/PR-Review-Agent.git"
PROJECT_DIR = "/content/PR-Review-Agent"

project_path = pathlib.Path(PROJECT_DIR)
if REPO_URL and not project_path.exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
elif REPO_URL and project_path.exists():
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    print("Set REPO_URL above, or upload the repository to /content/PR-Review-Agent manually.")

if project_path.exists():
    os.chdir(PROJECT_DIR)
    print("Working directory:", os.getcwd())
    subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)


## 3. Install Dependencies

The project uses heuristic tools for deterministic training and tests. This avoids depending on external analyzers during GRPO.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!python -m pip install -U pip
!python -m pip install -e . --no-deps
!python -m pip install \
  pydantic httpx fastapi uvicorn pyyaml \
  trl peft bitsandbytes accelerate transformers datasets torch \
  matplotlib pandas

os.environ["PR_REVIEW_TOOL_BACKEND"] = "heuristic"
os.environ["PYTHONUNBUFFERED"] = "1"


## 4. Deterministic Sanity Checks

Run these before using GPU time. They verify the task bank, tests, reward path, and replayable training rows.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!PR_REVIEW_TOOL_BACKEND=heuristic pytest
!python tasks/build_task_bank.py --print-summary

!PR_REVIEW_TOOL_BACKEND=heuristic python - <<'PY'
from train.train_config import TrainingConfig
from train.grpo_train import build_training_state_rows

cfg = TrainingConfig.for_cpu()
cfg.tasks_file = 'all'
cfg.training_task_limit = 78
rows = build_training_state_rows(cfg)
print({'training_rows': len(rows), 'first_task': rows[0]['task_id'], 'has_route': 'route' in rows[0]})
PY


## 5. Optional Baseline Evaluation

This gives you a reference before training. The heuristic baseline is hand-coded and should not be treated as an untrained model.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!PR_REVIEW_TOOL_BACKEND=heuristic python benchmarks/evaluate_baselines.py \
  --task-bank all \
  --task-loader-mode short \
  --output rewards/baseline_eval_colab.json

!python - <<'PY'
import json
from pathlib import Path
path = Path('rewards/baseline_eval_colab.json')
data = json.loads(path.read_text())
for name, summary in data.items():
    print(name, {k: summary[k] for k in ['accuracy', 'mean_episode_return', 'duplicate_tool_rate', 'early_submit_rate', 'evidence_backed_verdict_rate']})
PY


## 6. GRPO Training

Start with the T4 preset on Colab free/pro. For A100, switch `PRESET` to `a100`.

This trains `Qwen/Qwen3-1.7B` with QLoRA. Keep `PR_REVIEW_TOOL_BACKEND=heuristic` for deterministic rewards.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

PRESET = "t4"          # "t4" for Colab T4, "a100" for A100
TASK_LIMIT = 40         # use 78 for full-bank A100 training
EPOCHS = 1              # increase to 2-3 after a successful smoke run
NUM_GENERATIONS = 4     # GRPO needs >=2; lower this if Colab OOMs
OUTPUT_DIR = "./grpo_checkpoint_colab"

!PR_REVIEW_TOOL_BACKEND=heuristic python train/grpo_train.py \
  --preset {PRESET} \
  --task-bank all \
  --task-loader-mode short \
  --task-limit {TASK_LIMIT} \
  --epochs {EPOCHS} \
  --num-generations {NUM_GENERATIONS} \
  --output-dir {OUTPUT_DIR} \
  --report-to none


## 7. Evaluate the Checkpoint

Run this after training completes. Use a small limit first, then remove or increase it.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

EVAL_LIMIT = 20  # set 0 for all tasks

!PR_REVIEW_TOOL_BACKEND=heuristic python benchmarks/evaluate_trained_model.py \
  --checkpoint {OUTPUT_DIR} \
  --base-model Qwen/Qwen3-1.7B \
  --task-bank all \
  --task-loader-mode short \
  --limit {EVAL_LIMIT} \
  --output rewards/trained_eval_colab.json

!python - <<'PY'
import json
from pathlib import Path
path = Path('rewards/trained_eval_colab.json')
data = json.loads(path.read_text())
keys = ['episodes', 'accuracy', 'mean_episode_return', 'duplicate_tool_rate', 'early_submit_rate', 'evidence_backed_verdict_rate']
print({k: data.get(k) for k in keys})
PY


## 8. Optional Inference Packet Smoke Test

This checks CLI output from the trained checkpoint on a few tasks. For local Ollama checks, use `inference.py --model ollama:qwen2.5:7b` on your machine instead of Colab.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!PR_REVIEW_TOOL_BACKEND=heuristic python inference.py \
  --model Qwen/Qwen3-1.7B \
  --checkpoint {OUTPUT_DIR} \
  --limit 3 \
  --output-format packet


## 9. Package Artifacts

Save the LoRA checkpoint, training log, and evaluation JSONs. Download the zip from the Colab file browser or use the download cell.


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!zip -r pr_review_training_artifacts.zip \
  grpo_checkpoint_colab \
  rewards/baseline_eval_colab.json \
  rewards/trained_eval_colab.json \
  2>/dev/null || true

try:
    from google.colab import files
    files.download('pr_review_training_artifacts.zip')
except Exception as exc:
    print('Download manually from:', pathlib.Path('pr_review_training_artifacts.zip').resolve())
    print(exc)


## Notes on Readiness

Use this notebook first for smoke training. Before a longer run, compare trained results against `baseline_eval_colab.json` and check:

- action validity improves
- duplicate rate stays low
- early submit rate stays low
- evidence-backed verdict rate improves
- terminal accuracy improves or the model becomes cheaper/more evidence-aware than baseline
